# RODA.MOTORKU - Pemodelan Machine Learning Estimasi Harga Motor Bekas
## Perbandingan Algoritma: Linear Regression, Decision Tree Regressor, & Random Forest Regressor

**Peneliti / Pengembang:** Tim Capstone Project Roda.Motorku 2026  
**Tujuan:** Membangun sistem inferensi valuasi harga pasar motor bekas UMKM showroom berdasarkan 8 fitur spesifikasi teknis dan finansial (Sesuai Bab III & Bab IV Laporan Capstone).

### 1. Import Library & Modul Analisis Data

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

print("Modul berhasil dimuat!")

### 2. Memuat Dataset 187 Motor Bekas (Tabel 3.1)

In [ ]:
# Memuat dataset Excel
excel_path = 'dataset_motor_bekas_187-v2.xlsx'
zip_path = 'dataset_motor_bekas_187-v2.zip'

if os.path.exists(excel_path):
    df = pd.read_excel(excel_path)
elif os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        with z.open(excel_path) as f:
            df = pd.read_excel(f)

print(f"Dimensi Dataset: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

### 3. Pra-Pemrosesan Data (Data Preprocessing & Encoding)
- 8 Fitur Masukan: `merek`, `tahun`, `kelengkapan_dokumen`, `kilometer`, `harga_beli`, `kondisi_mesin`, `domisili_samsat`, `cc`
- 1 Target: `harga_jual`
- One-Hot Encoding pada fitur kategorikal
- Split Data Latih : Data Uji (80% : 20%)

In [ ]:
# Standarisasi penamaan kolom
df.columns = [str(c).strip().lower().replace(' ', '_') for c in df.columns]
col_map = {
    'merk_motor': 'merek',
    'merk': 'merek',
    'tahun_pembuatan': 'tahun',
    'kelengkapan': 'kelengkapan_dokumen',
    'odometer': 'kilometer',
    'samsat': 'domisili_samsat',
    'kapasitas_mesin': 'cc'
}
df = df.rename(columns=col_map)

features = ['merek', 'tahun', 'kelengkapan_dokumen', 'kilometer', 'harga_beli', 'kondisi_mesin', 'domisili_samsat', 'cc']
target = 'harga_jual'

X = df[features]
y = df[target]

# 80:20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Data Latih (80%): {X_train.shape[0]} unit")
print(f"Data Uji   (20%): {X_test.shape[0]} unit")

### 4. Pelatihan & Evaluasi 3 Algoritma Regresi

In [ ]:
cat_cols = ['merek', 'kelengkapan_dokumen', 'kondisi_mesin', 'domisili_samsat']
num_cols = ['tahun', 'kilometer', 'harga_beli', 'cc']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', StandardScaler(), num_cols)
])

models = {
    'Linear Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', Ridge(alpha=1.0))
    ]),
    'Decision Tree Regressor': Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', DecisionTreeRegressor(max_depth=8, min_samples_split=4, random_state=42))
    ]),
    'Random Forest Regressor': Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(n_estimators=150, max_features='sqrt', random_state=42))
    ])
}

eval_results = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    eval_results.append({
        'Algoritma': name,
        'MAE (Rp)': f"Rp {mae:,.0f}",
        'RMSE (Rp)': f"Rp {rmse:,.0f}",
        'R2 Score (%)': f"{r2 * 100:.2f}%"
    })

pd.DataFrame(eval_results)

### 5. Simpan Model Pipeline Terbaik untuk Integrasi Web

In [ ]:
# Menyimpan model terbaik ke file .pkl
best_pipe = models['Random Forest Regressor']
joblib.dump(best_pipe, 'model_terbaik_roda_motorku.pkl')
print("Model Random Forest Regressor berhasil diekspor ke: model_terbaik_roda_motorku.pkl")